In [1]:
import time
import numpy as np
import polanalyser as pa

np.random.seed(0)

list_filepath = [
    "pbrdf/1_spectralon_mitsuba/1_spectralon_raw.pbsdf",
    "pbrdf/2_white_billiard_mitsuba/2_white_billiard_raw.pbsdf",
    "pbrdf/5_brass_mitsuba/5_brass_raw.pbsdf",
]

num_samples = 361 * 91 * 91 * 5
# num_samples = 100000  # Reduce the number of samples for testing

for filepath in list_filepath:
    print("=" * 60)

    # Load a pBRDF table
    print(filepath)
    pbrdf_table = pa.pbrdf.load(filepath)
    M = pbrdf_table["M"]  # (..., 4, 4)
    print(f"M (original): {M.shape}, {M.dtype}")

    # Remove NaN values
    M = M[~np.any(np.isnan(M), axis=(-2, -1))]  # (:, 4, 4)
    print(f"M (removed) : {M.shape}, {M.dtype}")

    # Randomly sample a subset of the data
    if len(M) > num_samples:
        M = M[np.random.choice(M.shape[0], num_samples, replace=False)]  # (num_samples, 4, 4)
    print(f"M (sampled) : {M.shape}, {M.dtype}")

    # Givens-Kostinski method
    time_start = time.time()
    ismueller_gk = pa.ismueller(M)
    time_end = time.time()
    num_valid_gk = np.sum(ismueller_gk)
    num_elements_gk = ismueller_gk.size
    print(f"Givens-Kostinski {time_end - time_start:.3f} sec: {num_valid_gk}/{num_elements_gk} ({num_valid_gk / num_elements_gk:.2%})")

pbrdf/1_spectralon_mitsuba/1_spectralon_raw.pbsdf
M (original): (361, 91, 91, 5, 4, 4), float32
M (removed) : (9793049, 4, 4), float32
M (sampled) : (9793049, 4, 4), float32
Givens-Kostinski 17.897 sec: 9116898/9793049 (93.10%)
pbrdf/2_white_billiard_mitsuba/2_white_billiard_raw.pbsdf
M (original): (361, 91, 91, 5, 4, 4), float32
M (removed) : (9654216, 4, 4), float32
M (sampled) : (9654216, 4, 4), float32
Givens-Kostinski 17.498 sec: 7902732/9654216 (81.86%)
pbrdf/5_brass_mitsuba/5_brass_raw.pbsdf
M (original): (361, 91, 91, 5, 4, 4), float32
M (removed) : (9559715, 4, 4), float32
M (sampled) : (9559715, 4, 4), float32
Givens-Kostinski 18.814 sec: 2255632/9559715 (23.60%)
